# Pipeline Generativo de Descoberta de Inibidores — PROT (SLC6A7)

Pipeline computacional para geração e triagem de candidatos a inibidores do transportador de prolina PROT (SLC6A7), desenvolvido como parte do projeto de mestrado no Laboratório de Neurofarmacologia e Neuroquímica (UFG) / NeuroSpin.

**Etapas cobertas neste notebook** (metodologia final, conforme relatório técnico):

1. Carregamento dos dados de referência e das moléculas geradas pelo REINVENT4
2. Filtragem por drug-likeness (regras de Lipinski + filtro PAINS)
3. *Applicability Domain* (AD) via embeddings ChemBERTa + kNN
4. Score composto (NLL + distância ao AD) e seleção final dos dois grupos de candidatos
5. Análise de scaffold consenso (Murcko + Maximum Common Substructure)

**Não incluído neste repositório:**
- O modelo generativo REINVENT4 em si (transfer learning) — executado externamente
- O co-folding estrutural com Boltz-2 — executado externamente, resultados no relatório completo
- A ferramenta de scoring de afinidade/seletividade (DTA) — peça central da dissertação de mestrado, mantida privada por ora

> Nota metodológica: uma abordagem inicial de AD baseada em similaridade de Tanimoto (ECFP4) combinada a um ensemble QSAR (Random Forest + Ridge + SVR) foi testada, mas descontinuada por instabilidade do modelo entre folds de validação cruzada (R² = 0,24–0,32). A avaliação de afinidade foi delegada ao co-folding estrutural com o Boltz-2, e o AD final passou a ser calculado no espaço latente do ChemBERTa.

## 0. Requisitos

In [ ]:
# pip install rdkit pandas numpy scikit-learn matplotlib transformers torch tqdm cairosvg

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from collections import defaultdict

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Descriptors, rdMolDescriptors, FilterCatalog,
    rdFingerprintGenerator, Draw
)
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from rdkit.Chem.BRICS import BRICSDecompose
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFMCS import FindMCS, MCSParameters, AtomCompare, BondCompare
from rdkit.Chem.Draw import rdMolDraw2D

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


## 1. Dados de entrada

Este notebook espera dois arquivos CSV em `data/`:

| Arquivo | Conteúdo | Colunas esperadas |
|---|---|---|
| `prot_ligands.csv` | 50 ligantes de referência do PROT (dataset de treino) | `SMILES`, `pChEMBL` (ou `pActivity`/`pIC50`/`pKi`) |
| `generated_prot.csv` | Saída do REINVENT4 (744 moléculas geradas por transfer learning) | `SMILES`, `SMILES_state`, `NLL` |

Ajuste os caminhos abaixo conforme sua estrutura de diretórios.

In [ ]:
DATA_DIR = Path("data")
REF_CSV  = DATA_DIR / "prot_ligands.csv"
GEN_CSV  = DATA_DIR / "generated_prot.csv"
OUT_DIR  = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


def load_reference(path):
    """Carrega e valida o dataset de referência (50 ligantes conhecidos do PROT)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if "smiles" in cl:
            col_map[c] = "SMILES"
        elif any(k in cl for k in ["pchembl", "activity", "pic50", "pki"]):
            col_map[c] = "pActivity"
    df.rename(columns=col_map, inplace=True)

    valid = []
    for _, row in df.iterrows():
        mol = Chem.MolFromSmiles(str(row["SMILES"]).strip())
        if mol is None:
            continue
        valid.append({"SMILES": Chem.MolToSmiles(mol),
                       "pActivity": float(row["pActivity"]), "mol": mol})
    result = pd.DataFrame(valid)
    print(f"Referência: {len(result)} compostos válidos")
    return result


def load_generated(path):
    """Carrega e valida a saída do REINVENT4 (SMILES_state == 1 = molécula válida)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    valid = []
    for _, row in df.iterrows():
        if int(row.get("SMILES_state", 1)) != 1:
            continue
        mol = Chem.MolFromSmiles(str(row["SMILES"]).strip())
        if mol is None:
            continue
        valid.append({"SMILES": Chem.MolToSmiles(mol),
                       "NLL": float(row["NLL"]), "mol": mol})
    result = pd.DataFrame(valid).drop_duplicates(subset="SMILES").reset_index(drop=True)
    print(f"Gerados (REINVENT4): {len(result)} compostos válidos")
    return result


ref_df = load_reference(REF_CSV)
gen_df = load_generated(GEN_CSV)


## 2. Filtragem por drug-likeness (Lipinski + PAINS)

Remove moléculas já presentes no dataset de treino, compostos muito pequenos, fora das regras de Lipinski, e estruturas com alertas PAINS. Este funil reduz as 744 moléculas geradas a um subconjunto drug-like (contexto do relatório: 744 → 618).

In [ ]:
MW_MAX, LOGP_MAX, HBD_MAX, HBA_MAX, MIN_ATOMS = 600, 6.0, 5, 12, 15


def compute_descriptors(mol):
    return {
        "MW":         round(Descriptors.MolWt(mol), 2),
        "LogP":       round(Descriptors.MolLogP(mol), 3),
        "TPSA":       round(Descriptors.TPSA(mol), 2),
        "HBD":        rdMolDescriptors.CalcNumHBD(mol),
        "HBA":        rdMolDescriptors.CalcNumHBA(mol),
        "RotBonds":   rdMolDescriptors.CalcNumRotatableBonds(mol),
        "ArRings":    rdMolDescriptors.CalcNumAromaticRings(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
        "QED":        round(Descriptors.qed(mol), 4),
    }


def remove_known(gen_df, ref_df):
    known = set(ref_df["SMILES"].tolist())
    before = len(gen_df)
    out = gen_df[~gen_df["SMILES"].isin(known)].reset_index(drop=True)
    print(f"Removidos {before - len(out)} compostos já presentes na referência")
    return out


def apply_drug_likeness_filters(gen_df):
    pains_params = FilterCatalogParams()
    pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    pains_catalog = FilterCatalog.FilterCatalog(pains_params)

    rows, reasons = [], defaultdict(int)
    for _, row in gen_df.iterrows():
        mol, desc = row["mol"], compute_descriptors(row["mol"])
        if desc["HeavyAtoms"] < MIN_ATOMS: reasons["tamanho_minimo"] += 1; continue
        if desc["MW"]   > MW_MAX:          reasons["MW"]   += 1; continue
        if desc["LogP"] > LOGP_MAX:        reasons["LogP"] += 1; continue
        if desc["HBD"]  > HBD_MAX:         reasons["HBD"]  += 1; continue
        if desc["HBA"]  > HBA_MAX:         reasons["HBA"]  += 1; continue
        if pains_catalog.HasMatch(mol):    reasons["PAINS"] += 1; continue
        rows.append({**row.to_dict(), **desc})

    result = pd.DataFrame(rows).reset_index(drop=True)
    print(f"Compostos após filtros de drug-likeness: {len(result)}")
    for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
        print(f"  excluídos por {reason}: {count}")
    return result


gen_known_removed = remove_known(gen_df, ref_df)
gen_drug_like = apply_drug_likeness_filters(gen_known_removed)


## 3. Applicability Domain — embeddings ChemBERTa

O AD final é calculado no espaço latente do **ChemBERTa** (`seyonec/ChemBERTa-zinc-base-v1`), capturando contexto químico mais rico do que fingerprints binários. É aplicado sobre o conjunto completo de **744 moléculas geradas** (não sobre o subconjunto filtrado por drug-likeness da seção anterior) — essa é a metodologia usada nos resultados finais do relatório.

O AD é definido por distância média aos *k* vizinhos mais próximos (k=5) no espaço de referência; o threshold é o percentil 95 das distâncias intra-treino.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

MODEL_NAME  = "seyonec/ChemBERTa-zinc-base-v1"
K_NEIGHBORS = 5
PERCENTILE  = 95
BATCH_SIZE  = 32
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
chemberta_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
chemberta_model.eval()


def get_embeddings(smiles_list, batch_size=BATCH_SIZE, desc="Embeddings"):
    """Embedding do token [CLS] para cada SMILES — agrega o contexto global da sequência."""
    all_emb = []
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i: i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                         max_length=128, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = chemberta_model(**enc)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_emb)


emb_ref = normalize(get_embeddings(ref_df["SMILES"].tolist(), desc="Ref"))
emb_gen = normalize(get_embeddings(gen_df["SMILES"].tolist(), desc="Gen"))

# Threshold: percentil 95 das distâncias médias intra-treino (k+1 para excluir auto-distância)
knn_cal = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric="euclidean", n_jobs=-1).fit(emb_ref)
dist_ref = knn_cal.kneighbors(emb_ref)[0][:, 1:]
threshold = np.percentile(dist_ref.mean(axis=1), PERCENTILE)
print(f"AD threshold (percentil {PERCENTILE} intra-treino): {threshold:.4f}")

knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="euclidean", n_jobs=-1).fit(emb_ref)
dist_gen = knn.kneighbors(emb_gen)[0].mean(axis=1)

gen_df["ad_distance_chemberta"] = np.round(dist_gen, 6)
gen_df["inside_ad_chemberta"] = dist_gen <= threshold

n_in = gen_df["inside_ad_chemberta"].sum()
print(f"Dentro do AD: {n_in}/{len(gen_df)} ({100*n_in/len(gen_df):.1f}%)")


### Visualização — PCA do espaço latente + distribuição de distâncias

In [ ]:
pca = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(np.vstack([emb_ref, emb_gen]))
proj_ref, proj_gen = proj[:len(emb_ref)], proj[len(emb_ref):]
mask_in = gen_df["inside_ad_chemberta"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].scatter(*proj_gen[~mask_in].T, c="#f28b82", s=16, alpha=0.4, label="Fora do AD")
axes[0].scatter(*proj_gen[mask_in].T,  c="#34a853", s=20, alpha=0.6, label="Dentro do AD")
axes[0].scatter(*proj_ref.T, c="#1a73e8", s=50, edgecolors="white", label="Referência PROT")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title("Espaço ChemBERTa (PCA 2D)")
axes[0].legend(fontsize=9)

axes[1].hist(dist_ref.mean(axis=1), bins=15, color="#1a73e8", alpha=0.7, density=True, label="Referência")
axes[1].hist(dist_gen[mask_in],  bins=30, color="#34a853", alpha=0.6, density=True, label="Geradas – dentro AD")
axes[1].hist(dist_gen[~mask_in], bins=30, color="#f28b82", alpha=0.5, density=True, label="Geradas – fora AD")
axes[1].axvline(threshold, color="black", ls="--", label=f"Threshold = {threshold:.3f}")
axes[1].set_xlabel("Distância média aos k vizinhos")
axes[1].set_title("Distribuição de distâncias")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "ad_chemberta_prot.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Score composto e seleção final

Score = 0,60 × NLL⁻¹ (confiança generativa do REINVENT4) + 0,40 × distância ChemBERTa invertida, com penalidade multiplicativa por fragmento BRICS prejudicial identificado no dataset de referência.

Após um filtro de novidade mínima (Tanimoto ECFP4 < 0,85 frente ao treino, evitando cópias quase idênticas), os candidatos dentro do AD são ordenados por score e divididos em dois grupos de 30:

- **Grupo Otimizações** (`PROT_otim_01–30`): maiores scores, mais próximos do espaço de referência — modificações incrementais ao scaffold dominante
- **Grupo Novos Compostos** (`PROT_novo_01–30`): menores scores ainda dentro do AD — maior novidade estrutural

In [ ]:
from rdkit.Chem import AllChem

TANIMOTO_NOVELTY_MAX = 0.85   # exclui candidatos "cópias" do treino
N_SELECT              = 30
SAR_PENALTY           = 0.10

# Fragmentos BRICS prejudiciais identificados na análise SAR do dataset de referência
HARMFUL_SMARTS = [
    "[16*]c1ccccc1[16*]",   # bifenila não substituída
    "[14*]c1ccccn1",        # piridina terminal
    "[16*]c1ccccc1",        # fenila simples
    "[16*]c1ccc(F)cc1",     # flúor para
    "[3*]O[3*]",            # éter/oxigênio ligante
]

candidates = gen_df[gen_df["inside_ad_chemberta"]].copy().reset_index(drop=True)
print(f"Candidatos dentro do AD: {len(candidates)}")


def to_fp(smi):
    mol = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) if mol else None


fps_ref = [to_fp(s) for s in ref_df["SMILES"]]
fps_cand = [to_fp(s) for s in candidates["SMILES"]]
candidates["max_tanimoto"] = [
    max(DataStructs.BulkTanimotoSimilarity(fp, fps_ref)) if fp else 1.0 for fp in fps_cand
]
before = len(candidates)
candidates = candidates[candidates["max_tanimoto"] < TANIMOTO_NOVELTY_MAX].reset_index(drop=True)
print(f"Removidos por novidade insuficiente (Tanimoto >= {TANIMOTO_NOVELTY_MAX}): {before - len(candidates)}")

# Penalidade SAR
import re
harmful_mols = [Chem.MolFromSmarts(re.sub(r'\[\d+\*\]', '[#6,#7,#8,#16,#9,#17,#35]', s))
                for s in HARMFUL_SMARTS]
harmful_mols = [m for m in harmful_mols if m]
candidates["n_harmful"] = candidates["SMILES"].apply(
    lambda smi: sum(Chem.MolFromSmiles(smi).HasSubstructMatch(h) for h in harmful_mols)
)
candidates["sar_penalty"] = candidates["n_harmful"].apply(lambda n: max(0.50, 1.0 - n * SAR_PENALTY))

# Score composto
nll_inv = 1 / (candidates["NLL"] + 1e-6)
candidates["norm_NLL_inv"] = (nll_inv - nll_inv.min()) / (nll_inv.max() - nll_inv.min())
ad_inv = 1 - candidates["ad_distance_chemberta"]
candidates["norm_AD_inv"] = (ad_inv - ad_inv.min()) / (ad_inv.max() - ad_inv.min())
candidates["score"] = (0.60 * candidates["norm_NLL_inv"] + 0.40 * candidates["norm_AD_inv"]) * candidates["sar_penalty"]
candidates = candidates.sort_values("score", ascending=False).reset_index(drop=True)

# Grupo Otimizações (30 maiores scores) e Grupo Novos Compostos (30 menores scores)
grupo_otimizacoes = candidates.head(N_SELECT).copy()
grupo_otimizacoes.insert(0, "ID", [f"PROT_otim_{i+1:02d}" for i in range(len(grupo_otimizacoes))])

grupo_novos = candidates.tail(N_SELECT).sort_values("score", ascending=True).reset_index(drop=True).copy()
grupo_novos.insert(0, "ID", [f"PROT_novo_{i+1:02d}" for i in range(len(grupo_novos))])

cols = ["ID", "SMILES", "NLL", "ad_distance_chemberta", "max_tanimoto",
        "n_harmful", "sar_penalty", "score"]
grupo_otimizacoes[cols].to_csv(OUT_DIR / "grupo_otimizacoes.csv", index=False)
grupo_novos[cols].to_csv(OUT_DIR / "grupo_novos_compostos.csv", index=False)

print(f"\nGrupo Otimizações — score médio: {grupo_otimizacoes['score'].mean():.4f}, "
      f"d̄ ChemBERTa médio: {grupo_otimizacoes['ad_distance_chemberta'].mean():.4f}")
print(f"Grupo Novos Compostos — score médio: {grupo_novos['score'].mean():.4f}, "
      f"d̄ ChemBERTa médio: {grupo_novos['ad_distance_chemberta'].mean():.4f}")


## 5. Análise de scaffold consenso (Murcko + MCS)

Análise complementar e independente do funil de seleção acima: identifica o núcleo farmacofórico do dataset de referência combinando o **scaffold de Bemis–Murcko** mais frequente com a **Maximum Common Substructure (MCS)** entre todos os 50 compostos. A sobreposição das duas abordagens separa:

- **Regiões conservadas** (Murcko ∩ MCS) — provável núcleo farmacofórico essencial
- **Pontos de diversificação** (Murcko fora do MCS) — posições já exploradas estruturalmente no dataset, prioritárias para modificações racionais

In [ ]:
def get_murcko_scaffolds(df):
    scaffold_data, scaffold_mols = defaultdict(list), {}
    for _, row in df.iterrows():
        try:
            sc_mol = MurckoScaffold.GetScaffoldForMol(row["mol"])
            sc_smi = Chem.MolToSmiles(sc_mol)
            scaffold_data[sc_smi].append(row["pActivity"])
            scaffold_mols.setdefault(sc_smi, sc_mol)
        except Exception:
            pass
    rows = [{"scaffold_smiles": smi, "frequency": len(acts),
             "mean_pActivity": round(np.mean(acts), 3), "mol": scaffold_mols[smi]}
            for smi, acts in scaffold_data.items()]
    sc_df = pd.DataFrame(rows).sort_values("frequency", ascending=False).reset_index(drop=True)
    print(f"Scaffolds únicos: {len(sc_df)}")
    print(f"Dominante: {sc_df.iloc[0]['scaffold_smiles']}  "
          f"({sc_df.iloc[0]['frequency']} compostos, "
          f"{100*sc_df.iloc[0]['frequency']/len(df):.0f}% do dataset)")
    return sc_df


def compute_mcs(df, timeout=60):
    params = MCSParameters()
    params.AtomTyper, params.BondTyper = AtomCompare.CompareElements, BondCompare.CompareOrder
    params.Timeout, params.Threshold = timeout, 0.8
    result = FindMCS(df["mol"].tolist(), params)
    print(f"MCS: {result.numAtoms} átomos, {result.numBonds} ligações")
    return result, Chem.MolFromSmarts(result.smartsString)


def annotate_scaffold(murcko_mol, mcs_mol):
    match = murcko_mol.GetSubstructMatch(mcs_mol)
    conserved = set(match)
    diverse = set(range(murcko_mol.GetNumAtoms())) - conserved
    print(f"Átomos conservados: {len(conserved)}  |  Pontos de diversificação: {len(diverse)}")
    return list(conserved), list(diverse)


sc_df = get_murcko_scaffolds(ref_df)
murcko_mol = sc_df.iloc[0]["mol"]
mcs_result, mcs_mol = compute_mcs(ref_df)

if mcs_mol is not None and murcko_mol.HasSubstructMatch(mcs_mol):
    conserved_atoms, diverse_atoms = annotate_scaffold(murcko_mol, mcs_mol)
else:
    print("MCS não faz match direto no Murcko dominante — inspecionar manualmente.")

# Renderização do scaffold anotado (conservado = azul, diversificação = vermelho)
BLUE, RED = (0.196, 0.400, 0.678), (0.851, 0.353, 0.188)
drawer = rdMolDraw2D.MolDraw2DSVG(900, 650)
drawer.DrawMolecule(
    murcko_mol,
    highlightAtoms=conserved_atoms + diverse_atoms,
    highlightAtomColors={**{i: BLUE for i in conserved_atoms}, **{i: RED for i in diverse_atoms}},
)
drawer.FinishDrawing()
(OUT_DIR / "scaffold_annotated.svg").write_text(drawer.GetDrawingText())
print(f"\nScaffold anotado salvo em {OUT_DIR / 'scaffold_annotated.svg'}")


## Próximos passos (fora deste notebook)

1. **Co-folding estrutural com Boltz-2** — avaliação da compatibilidade dos 60 candidatos (Grupo Otimizações + Grupo Novos Compostos) com o sítio de ligação do PROT. Executado externamente; resultados completos no relatório técnico.
2. **Modelo de afinidade/seletividade (DTA)** — ferramenta de scoring standalone, peça central da dissertação de mestrado em andamento, mantida privada nesta etapa do projeto.